# Airbnb Price Prediction — V2 (Target: R² ≥ 0.80)
## Kya improve kiya:
- ✅ Advanced Neighbourhood clustering (lat/lon based geo-clusters)
- ✅ City × Room_type mean price (strong signal)
- ✅ Text-based features: name keywords (luxury, cozy, spacious, etc.)
- ✅ Amenity interaction features (luxury_bundle, essentials_score)
- ✅ Review-based quality signals
- ✅ CatBoost + XGBoost + LightGBM 3-model ensemble
- ✅ Optuna hyperparameter tuning (optional cell)
- ✅ Pseudo-label boosting idea

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')

Libraries loaded!


In [2]:
df = pd.read_csv('train_data.csv', encoding='latin1')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (74111, 29)


,id,log_price,property_type,room_type,amenities,accommodates,bathrooms,bed_type,cancellation_policy,cleaning_fee,...,latitude,longitude,name,neighbourhood,number_of_reviews,review_scores_rating,thumbnail_url,zipcode,bedrooms,beds
0,6901257,5.010635,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",3,1.0,Real Bed,strict,True,...,40.696524,-73.991617,Beautiful brownstone 1-bedroom,Brooklyn Heights,2,100.0,https://a0.muscache.com/im/pictures/6d7cbbf7-c...,11201,1.0,1.0
1,6304928,5.129899,Apartment,Entire home/apt,"{""Wireless Internet"",""Air conditioning"",Kitche...",7,1.0,Real Bed,strict,True,...,40.766115,-73.989040,Superb 3BR Apt Located Near Times Square,Hell's Kitchen,6,93.0,https://a0.muscache.com/im/pictures/348a55fe-4...,10019,3.0,3.0
2,7919400,4.976734,Apartment,Entire home/apt,"{TV,""Cable TV"",""Wireless Internet"",""Air condit...",5,1.0,Real Bed,moderate,True,...,40.808110,-73.943756,The Garden Oasis,Harlem,10,92.0,https://a0.muscache.com/im/pictures/6fae5362-9...,10027,1.0,3.0


## Step 1: Basic Feature Engineering (Same as Before)

In [3]:
# ---------- Amenities ----------
if 'amenities' in df.columns:
    s = df['amenities'].fillna('').astype(str).str.lower()
    df['has_wifi']          = s.str.contains('wireless internet|wifi').astype(int)
    df['has_ac']            = s.str.contains('air conditioning').astype(int)
    df['has_kitchen']       = s.str.contains('kitchen').astype(int)
    df['has_tv']            = s.str.contains('"tv"').astype(int)
    df['has_washer']        = s.str.contains('washer').astype(int)
    df['has_dryer']         = s.str.contains('dryer').astype(int)
    df['has_parking']       = s.str.contains('parking').astype(int)
    df['has_gym']           = s.str.contains('gym').astype(int)
    df['has_pool']          = s.str.contains('pool').astype(int)
    df['has_elevator']      = s.str.contains('elevator').astype(int)
    df['has_doorman']       = s.str.contains('doorman').astype(int)
    df['has_breakfast']     = s.str.contains('breakfast').astype(int)
    df['has_pets']          = s.str.contains('pets allowed').astype(int)
    df['has_hottub']        = s.str.contains('hot tub').astype(int)
    df['has_heating']       = s.str.contains('heating').astype(int)
    df['has_shampoo']       = s.str.contains('shampoo').astype(int)
    df['has_iron']          = s.str.contains('iron').astype(int)
    df['has_hangers']       = s.str.contains('hangers').astype(int)
    df['has_laptop_friendly']= s.str.contains('laptop friendly').astype(int)
    df['has_smoke_detector']= s.str.contains('smoke detector').astype(int)
    df['has_carbon_detector']= s.str.contains('carbon monoxide detector').astype(int)
    df['has_fire_extinguish']= s.str.contains('fire extinguisher').astype(int)
    df['has_first_aid']     = s.str.contains('first aid').astype(int)
    df['amenity_count']     = s.str.count(',') + 1
    df['amenity_count']     = df['amenity_count'].clip(0, 60)
    df['log_amenity_count'] = np.log1p(df['amenity_count'])
    
    # ✅ NEW: Bundled amenity scores (stronger signal than individual)
    df['safety_score'] = (
        df['has_smoke_detector'] + df['has_carbon_detector'] +
        df['has_fire_extinguish'] + df['has_first_aid']
    )
    df['luxury_bundle'] = (
        df['has_pool'] * 3 + df['has_gym'] * 2 + df['has_doorman'] * 2 +
        df['has_hottub'] * 2 + df['has_elevator'] + df['has_parking']
    )
    df['essentials_score'] = (
        df['has_wifi'] + df['has_kitchen'] + df['has_heating'] +
        df['has_washer'] + df['has_dryer'] + df['has_tv']
    )
    df['comfort_score'] = (
        df['has_iron'] + df['has_hangers'] + df['has_shampoo'] +
        df['has_laptop_friendly'] + df['has_breakfast']
    )
    print('Amenities done')

# ---------- Neighbourhood ----------
if 'neighbourhood' in df.columns:
    df['neighbourhood'] = df['neighbourhood'].fillna('Unknown')

# ---------- Zipcode ----------
if 'zipcode' in df.columns:
    df['zipcode_num'] = pd.to_numeric(df['zipcode'], errors='coerce').fillna(0).astype(int)

print(f'Shape after amenities: {df.shape}')

Amenities done
Shape after amenities: (74111, 59)


In [4]:
# ---------- Numeric fill ----------
num_fill_cols = ['bathrooms', 'bedrooms', 'beds', 'review_scores_rating',
                 'review_scores_accuracy', 'review_scores_cleanliness',
                 'review_scores_checkin', 'review_scores_communication',
                 'review_scores_location', 'review_scores_value']
for col in num_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

cat_fill_cols = ['host_has_profile_pic', 'host_identity_verified']
for col in cat_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

print('Nulls filled')

Nulls filled


## Step 2: 🔥 ADVANCED Feature Engineering (Main Improvement)

In [5]:
# ---------- Basic Ratio Features ----------
df['beds_per_person']    = df['beds']      / df['accommodates'].clip(1)
df['bath_per_person']    = df['bathrooms'] / df['accommodates'].clip(1)
df['bed_bath_ratio']     = df['beds']      / (df['bathrooms'] + 0.5)
df['room_per_person']    = df['bedrooms']  / df['accommodates'].clip(1)
df['beds_per_bedroom']   = df['beds'] / (df['bedrooms'] + 0.5)

# ---------- Log/Sqrt transforms ----------
df['log_reviews']        = np.log1p(df['number_of_reviews'])
df['log_accommodates']   = np.log1p(df['accommodates'])
df['sqrt_accommodates']  = np.sqrt(df['accommodates'])
df['sqrt_reviews']       = np.sqrt(df['number_of_reviews'])

# ---------- Rating flags ----------
df['high_rating']        = (df['review_scores_rating'] >= 95).astype(int)
df['low_rating']         = (df['review_scores_rating'] < 80).astype(int)
df['perfect_rating']     = (df['review_scores_rating'] == 100).astype(int)
df['new_listing']        = (df['number_of_reviews'] == 0).astype(int)
df['popular_listing']    = (df['number_of_reviews'] > 20).astype(int)
df['experienced_host']   = (df['number_of_reviews'] > 50).astype(int)

# ---------- ✅ NEW: Composite review score (weighted average) ----------
review_cols = ['review_scores_accuracy', 'review_scores_cleanliness',
               'review_scores_checkin', 'review_scores_communication',
               'review_scores_location', 'review_scores_value']
review_weights = [1.0, 1.5, 0.5, 0.5, 2.0, 1.5]  # location & value matter more
existing_review_cols = [c for c in review_cols if c in df.columns]
if existing_review_cols:
    weights = review_weights[:len(existing_review_cols)]
    df['composite_review_score'] = (
        sum(df[c] * w for c, w in zip(existing_review_cols, weights)) /
        sum(weights)
    )
    df['review_score_spread'] = df[existing_review_cols].std(axis=1).fillna(0)
    print(f'Composite review score created from {len(existing_review_cols)} sub-scores')

# ---------- Geo Features ----------
df['lat_lon_interact']   = df['latitude'] * df['longitude']
df['lat_rounded']        = (df['latitude']  * 10).round() / 10
df['lon_rounded']        = (df['longitude'] * 10).round() / 10

# ---------- ✅ NEW: KMeans Geo-Clusters (neighborhoods proxy) ----------
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

geo_features = df[['latitude', 'longitude']].fillna(0)
scaler_geo = StandardScaler()
geo_scaled = scaler_geo.fit_transform(geo_features)

# Try different cluster counts — 20-30 works well for Airbnb cities
for n_clusters in [20, 50]:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df[f'geo_cluster_{n_clusters}'] = kmeans.fit_predict(geo_scaled)

print(f'Geo clusters created')
print(f'Shape: {df.shape}')

Geo clusters created
Shape: (74111, 79)


In [6]:
# ---------- ✅ NEW: Text Features from listing NAME ----------
if 'name' in df.columns:
    name = df['name'].fillna('').astype(str).str.lower()
    
    # Luxury keywords → higher price signal
    luxury_words = ['luxury', 'penthouse', 'villa', 'mansion', 'private', 
                    'exclusive', 'premium', 'stunning', 'amazing', 'spectacular']
    cozy_words   = ['cozy', 'cosy', 'charming', 'quaint', 'cute', 'tiny']
    space_words  = ['spacious', 'large', 'huge', 'big', 'entire', 'rooftop']
    location_words = ['downtown', 'central', 'prime', 'heart', 'walking distance',
                      'beachfront', 'ocean view', 'city view', 'park view']
    
    df['name_has_luxury']   = name.str.contains('|'.join(luxury_words)).astype(int)
    df['name_has_cozy']     = name.str.contains('|'.join(cozy_words)).astype(int)
    df['name_has_spacious'] = name.str.contains('|'.join(space_words)).astype(int)
    df['name_has_location'] = name.str.contains('|'.join(location_words)).astype(int)
    df['name_length']       = name.str.len()
    df['name_word_count']   = name.str.split().str.len().fillna(0)
    df['name_has_number']   = name.str.contains(r'\d').astype(int)  # "3BR", "2 bed"
    
    print('Name text features done')

# ---------- ✅ NEW: Description features ----------
if 'description' in df.columns:
    desc = df['description'].fillna('').astype(str).str.lower()
    df['description_length']  = desc.str.len()
    df['desc_word_count']     = desc.str.split().str.len().fillna(0)
    df['desc_has_luxury']     = desc.str.contains('luxury|upscale|high-end').astype(int)
    df['desc_has_transport']  = desc.str.contains('subway|metro|bus|transport|walk').astype(int)
    print('Description features done')

Name text features done
Description features done


In [7]:
# ---------- Host Features ----------
# host_response_rate
df['host_response_rate_num'] = pd.to_numeric(
    df['host_response_rate'].astype(str).str.replace('%','').str.strip(),
    errors='coerce'
)
df['host_response_rate_num'] = df['host_response_rate_num'].fillna(
    df['host_response_rate_num'].median()
)

# host_since
df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
df['host_experience_years'] = (
    pd.Timestamp('2017-01-01') - df['host_since']
).dt.days / 365.0
df['host_experience_years'] = df['host_experience_years'].fillna(
    df['host_experience_years'].median()
)

# ✅ NEW: Host experience buckets
df['host_exp_bucket'] = pd.cut(
    df['host_experience_years'],
    bins=[-999, 0.5, 1, 2, 4, 999],
    labels=[0, 1, 2, 3, 4]
).astype(float).fillna(0)

# first_review / last_review
df['first_review'] = pd.to_datetime(df['first_review'], errors='coerce')
df['listing_age_days'] = (
    pd.Timestamp('2017-01-01') - df['first_review']
).dt.days.fillna(0)

df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['days_since_last_review'] = (
    pd.Timestamp('2017-01-01') - df['last_review']
).dt.days.fillna(0)

# ✅ NEW: Recency score (recent reviews = active listing)
df['is_recently_reviewed'] = (df['days_since_last_review'] < 90).astype(int)
df['is_stale_listing']     = (df['days_since_last_review'] > 365).astype(int)
df['log_listing_age']      = np.log1p(df['listing_age_days'])

# ✅ NEW: Reviews per day (activity metric)
df['reviews_per_day'] = df['number_of_reviews'] / (df['listing_age_days'].clip(1))
df['log_reviews_per_day'] = np.log1p(df['reviews_per_day'])

# ✅ NEW: host_listings_count
if 'host_listings_count' in df.columns:
    df['host_listings_count'] = df['host_listings_count'].fillna(1)
    df['is_multi_host']       = (df['host_listings_count'] > 1).astype(int)
    df['is_superhost_proxy']  = (df['host_listings_count'] > 5).astype(int)
    df['log_host_listings']   = np.log1p(df['host_listings_count'])

# ✅ NEW: Capacity categories
df['is_large_listing']   = (df['accommodates'] >= 6).astype(int)
df['is_studio']          = (df['bedrooms'] <= 1).astype(int)
df['is_whole_apt']       = (df['bedrooms'] >= 2).astype(int)
df['accommodates_sq']    = df['accommodates'] ** 2
df['bedrooms_sq']        = df['bedrooms'] ** 2

# ✅ NEW: min_nights interaction
if 'minimum_nights' in df.columns:
    df['minimum_nights'] = df['minimum_nights'].fillna(1).clip(1, 365)
    df['is_long_term']   = (df['minimum_nights'] >= 7).astype(int)
    df['is_short_term']  = (df['minimum_nights'] == 1).astype(int)
    df['log_min_nights'] = np.log1p(df['minimum_nights'])

print('Host + temporal features done!')
print(f'Shape: {df.shape}')

Host + temporal features done!
Shape: (74111, 105)


In [8]:
# ---------- Remove Outliers ----------
q_lo = df['log_price'].quantile(0.01)
q_hi = df['log_price'].quantile(0.99)
df   = df[(df['log_price'] >= q_lo) & (df['log_price'] <= q_hi)].copy()

print(f'After outlier removal: {len(df)} rows, {len(df.columns)} cols')

After outlier removal: 72651 rows, 105 cols


## Step 3: Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    'room_type', 'property_type', 'bed_type',
    'cancellation_policy', 'city', 'cleaning_fee',
    'instant_bookable', 'host_has_profile_pic',
    'host_identity_verified'
]
categorical_cols = [c for c in categorical_cols if c in df.columns]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

if 'neighbourhood' in df.columns:
    le_nb = LabelEncoder()
    df['neighbourhood_enc'] = le_nb.fit_transform(df['neighbourhood'].astype(str))

print('Label encoding done!')

Label encoding done!


## Step 4: Train/Test Split + Target Encoding (No Leakage)

In [10]:
from sklearn.model_selection import train_test_split

base_feature_cols = [c for c in df.columns
                     if c != 'log_price'
                     and c != 'neighbourhood'
                     and df[c].dtype != object]

X_base = df[base_feature_cols].copy()
y      = df['log_price'].copy()

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train_base.shape}, Test: {X_test_base.shape}')

Train: (58120, 104), Test: (14531, 104)


In [11]:
X_train = X_train_base.copy()
X_test  = X_test_base.copy()

global_mean = y_train.mean()
smoothing   = 10

# ✅ Extended target encoding columns (geo clusters added!)
target_encode_cols = ['city', 'room_type', 'property_type', 'cancellation_policy']
if 'geo_cluster_20' in X_train.columns:
    target_encode_cols += ['geo_cluster_20', 'geo_cluster_50']
if 'neighbourhood_enc' in X_train.columns:
    target_encode_cols += ['neighbourhood_enc']

target_encode_cols = [c for c in target_encode_cols if c in X_train.columns]

target_enc_maps = {}
for col in target_encode_cols:
    stats = pd.DataFrame({'col': X_train[col].values, 'target': y_train.values})\
              .groupby('col')['target'].agg(['mean','count'])
    stats['smoothed'] = (
        (stats['count'] * stats['mean'] + smoothing * global_mean) /
        (stats['count'] + smoothing)
    )
    col_map = stats['smoothed'].to_dict()
    target_enc_maps[col] = col_map
    X_train[f'{col}_te'] = X_train[col].map(col_map).fillna(global_mean)
    X_test[f'{col}_te']  = X_test[col].map(col_map).fillna(global_mean)
    print(f'{col}: {len(col_map)} values encoded')

# ✅ NEW: City × Room_type interaction target encoding
if 'city' in X_train.columns and 'room_type' in X_train.columns:
    X_train['city_room_key'] = X_train['city'].astype(str) + '_' + X_train['room_type'].astype(str)
    X_test['city_room_key']  = X_test['city'].astype(str)  + '_' + X_test['room_type'].astype(str)
    
    stats2 = pd.DataFrame({'col': X_train['city_room_key'].values, 'target': y_train.values})\
               .groupby('col')['target'].agg(['mean','count'])
    stats2['smoothed'] = (
        (stats2['count'] * stats2['mean'] + smoothing * global_mean) /
        (stats2['count'] + smoothing)
    )
    city_room_map = stats2['smoothed'].to_dict()
    X_train['city_room_te'] = X_train['city_room_key'].map(city_room_map).fillna(global_mean)
    X_test['city_room_te']  = X_test['city_room_key'].map(city_room_map).fillna(global_mean)
    X_train.drop('city_room_key', axis=1, inplace=True)
    X_test.drop('city_room_key', axis=1, inplace=True)
    print('city × room_type interaction TE done')

# ✅ NEW: City × property_type TE
if 'city' in X_train.columns and 'property_type' in X_train.columns:
    X_train['city_prop_key'] = X_train['city'].astype(str) + '_' + X_train['property_type'].astype(str)
    X_test['city_prop_key']  = X_test['city'].astype(str)  + '_' + X_test['property_type'].astype(str)
    stats3 = pd.DataFrame({'col': X_train['city_prop_key'].values, 'target': y_train.values})\
               .groupby('col')['target'].agg(['mean','count'])
    stats3['smoothed'] = (
        (stats3['count'] * stats3['mean'] + smoothing * global_mean) /
        (stats3['count'] + smoothing)
    )
    city_prop_map = stats3['smoothed'].to_dict()
    X_train['city_prop_te'] = X_train['city_prop_key'].map(city_prop_map).fillna(global_mean)
    X_test['city_prop_te']  = X_test['city_prop_key'].map(city_prop_map).fillna(global_mean)
    X_train.drop('city_prop_key', axis=1, inplace=True)
    X_test.drop('city_prop_key', axis=1, inplace=True)
    print('city × property_type interaction TE done')

# City std price
if 'city' in X_train.columns:
    city_std_map = y_train.groupby(X_train['city']).std().to_dict()
    X_train['city_std_price'] = X_train['city'].map(city_std_map).fillna(y_train.std())
    X_test['city_std_price']  = X_test['city'].map(city_std_map).fillna(y_train.std())

# Drop string cols
object_cols = X_train.select_dtypes(include=['object']).columns.tolist()
X_train = X_train.drop(columns=object_cols).fillna(0)
X_test  = X_test.drop(columns=object_cols).fillna(0)

print(f'\nFinal features: {X_train.shape[1]}')
print('Target encoding done!')

city: 6 values encoded
room_type: 3 values encoded
property_type: 33 values encoded
cancellation_policy: 5 values encoded
geo_cluster_20: 20 values encoded
geo_cluster_50: 50 values encoded
neighbourhood_enc: 606 values encoded
city × room_type interaction TE done
city × property_type interaction TE done

Final features: 108
Target encoding done!


In [12]:
# ✅ FIX: Sab non-numeric columns drop karo (object + datetime dono)
# XGBoost sirf int / float / bool accept karta hai

bad_cols = [
    col for col in X_train.columns
    if X_train[col].dtype.kind not in ('i', 'u', 'f', 'b')
]
print(f"Drop ho rahi columns ({len(bad_cols)}): {bad_cols}")

X_train = X_train.drop(columns=bad_cols, errors='ignore').fillna(0)
X_test  = X_test.drop(columns=bad_cols,  errors='ignore').fillna(0)

# Double-check — agar koi column abhi bhi bura hai toh force convert karo
still_bad = [c for c in X_train.columns
             if X_train[c].dtype.kind not in ('i','u','f','b')]
if still_bad:
    print(f"Force converting: {still_bad}")
    for c in still_bad:
        X_train[c] = pd.to_numeric(X_train[c], errors='coerce').fillna(0)
        X_test[c]  = pd.to_numeric(X_test[c],  errors='coerce').fillna(0)

print(f"\nFinal shape  : {X_train.shape}")
print(f"Dtypes check : {X_train.dtypes.value_counts().to_dict()}")
print("Ready for XGBoost!")

Drop ho rahi columns (3): ['first_review', 'host_since', 'last_review']

Final shape  : (58120, 105)
Dtypes check : {dtype('int64'): 65, dtype('float64'): 38, dtype('int32'): 2}
Ready for XGBoost!


In [13]:
import numpy as np

# Step 1: inf ko nan se replace karo
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# Step 2: nan ko column median se fill karo
X_train = X_train.fillna(X_train.median())
X_test  = X_test.fillna(X_train.median())  # test mein bhi train ka median use karo

# Step 3: verify — koi nan/inf nahi rehna chahiye
nan_train = X_train.isnull().sum().sum()
nan_test  = X_test.isnull().sum().sum()
inf_train = np.isinf(X_train.values).sum()
inf_test  = np.isinf(X_test.values).sum()

print(f"NaN  — Train: {nan_train}, Test: {nan_test}")
print(f"Inf  — Train: {inf_train}, Test: {inf_test}")

if nan_train == 0 and nan_test == 0 and inf_train == 0 and inf_test == 0:
    print("All clean! XGBoost chalega ab.")
else:
    print("Abhi bhi kuch columns mein problem hai:")
    bad = X_train.columns[X_train.isnull().any() | np.isinf(X_train).any()].tolist()
    print(bad)
    # Force fix karo
    X_train[bad] = X_train[bad].fillna(0).replace([np.inf, -np.inf], 0)
    X_test[bad]  = X_test[bad].fillna(0).replace([np.inf, -np.inf], 0)
    print("Force fixed!")

NaN  — Train: 0, Test: 0
Inf  — Train: 0, Test: 0
All clean! XGBoost chalega ab.


## 🔧 FIX: Correct Neighbourhood Target Encoding
**Bug was**: Cell 18 used encoded int indices; Cell 20 overwrote with wrong index alignment → correlation = -0.003
**Fix**: Use original string neighbourhood from df with proper index reset

In [14]:
# ============================================================
# FIX 1: Correct city_neighbourhood TE (main bug fix)
# Problem: prev cells used label-encoded int → shuffled meaning
# Solution: use original string neighbourhood from df
# ============================================================

global_mean = y_train.mean()

# Reset index agar shifted hai
X_train_idx = X_train.index
X_test_idx  = X_test.index

# Original string neighbourhood directly from df
nb_col = df['neighbourhood'].fillna('Unknown').astype(str)
city_col = df['city'].astype(str)  # already encoded as int, use as-is

# Combined key using original string neighbourhood
df['_city_nb_key'] = city_col + '_' + nb_col

train_keys = df.loc[X_train_idx, '_city_nb_key']
test_keys  = df.loc[X_test_idx,  '_city_nb_key']

smoothing = 10
stats = pd.DataFrame({'key': train_keys.values, 'target': y_train.values}) \
          .groupby('key')['target'].agg(['mean','count'])
stats['smoothed'] = (
    (stats['count'] * stats['mean'] + smoothing * global_mean) /
    (stats['count'] + smoothing)
)
city_nb_map = stats['smoothed'].to_dict()

X_train['city_nb_te'] = train_keys.map(city_nb_map).fillna(global_mean).values
X_test['city_nb_te']  = test_keys.map(city_nb_map).fillna(global_mean).values

corr = pd.Series(X_train['city_nb_te']).corr(y_train)
print(f'city_nb_te correlation: {corr:.4f}  (0.60+ expected)')

# ============================================================
# FIX 2: Correct zipcode TE (using original df zipcode string)
# ============================================================
if 'zipcode' in df.columns:
    df['_city_zip_key'] = city_col + '_' + df['zipcode'].fillna('00000').astype(str).str[:5]
    train_zip = df.loc[X_train_idx, '_city_zip_key']
    test_zip  = df.loc[X_test_idx,  '_city_zip_key']
    
    smoothing_zip = 5
    stats_z = pd.DataFrame({'key': train_zip.values, 'target': y_train.values}) \
                .groupby('key')['target'].agg(['mean','count'])
    stats_z['smoothed'] = (
        (stats_z['count'] * stats_z['mean'] + smoothing_zip * global_mean) /
        (stats_z['count'] + smoothing_zip)
    )
    zip_map = stats_z['smoothed'].to_dict()
    X_train['city_zip_te'] = train_zip.map(zip_map).fillna(global_mean).values
    X_test['city_zip_te']  = test_zip.map(zip_map).fillna(global_mean).values
    corr_z = pd.Series(X_train['city_zip_te']).corr(y_train)
    print(f'city_zip_te correlation: {corr_z:.4f}')

# Drop temp columns
df.drop(columns=['_city_nb_key', '_city_zip_key'], errors='ignore', inplace=True)
print('Neighbourhood TE fixed!')


city_nb_te correlation: 0.4779  (0.60+ expected)
city_zip_te correlation: 0.4945
Neighbourhood TE fixed!


In [15]:
# ============================================================
# FIX 3: Interaction features (corrected — only AFTER nb_te is fixed)
# ============================================================

# Basic interactions
X_train['accom_x_bath']      = X_train['accommodates'] * X_train['bathrooms']
X_test['accom_x_bath']       = X_test['accommodates']  * X_test['bathrooms']

X_train['bed_x_bath']        = X_train['bedrooms'] * X_train['bathrooms']
X_test['bed_x_bath']         = X_test['bedrooms']  * X_test['bathrooms']

X_train['luxury_x_accom']    = X_train['luxury_bundle'] * X_train['accommodates']
X_test['luxury_x_accom']     = X_test['luxury_bundle']  * X_test['accommodates']

X_train['room_te_x_accom']   = X_train['room_type_te'] * X_train['accommodates']
X_test['room_te_x_accom']    = X_test['room_type_te']  * X_test['accommodates']

X_train['city_room_x_accom'] = X_train['city_room_te'] * X_train['accommodates']
X_test['city_room_x_accom']  = X_test['city_room_te']  * X_test['accommodates']

X_train['city_room_x_bed']   = X_train['city_room_te'] * X_train['bedrooms']
X_test['city_room_x_bed']    = X_test['city_room_te']  * X_test['bedrooms']

# NEW: city_nb_te interactions (now that it's correct!)
X_train['nb_te_x_accom']     = X_train['city_nb_te'] * X_train['accommodates']
X_test['nb_te_x_accom']      = X_test['city_nb_te']  * X_test['accommodates']

X_train['nb_te_x_room']      = X_train['city_nb_te'] * X_train['room_type_te']
X_test['nb_te_x_room']       = X_test['city_nb_te']  * X_test['room_type_te']

# NEW: price-proxy features
X_train['te_per_person']     = X_train['city_nb_te'] / X_train['accommodates'].clip(1)
X_test['te_per_person']      = X_test['city_nb_te']  / X_test['accommodates'].clip(1)

# NEW: zip_te interactions
if 'city_zip_te' in X_train.columns:
    X_train['zip_x_accom']   = X_train['city_zip_te'] * X_train['accommodates']
    X_test['zip_x_accom']    = X_test['city_zip_te']  * X_test['accommodates']

# Final clean
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(X_train.median())
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(X_train.median())

# Check key correlations
key_feats = ['city_nb_te','city_zip_te','nb_te_x_accom','city_room_te','room_te_x_accom']
key_feats = [f for f in key_feats if f in X_train.columns]
corrs = pd.DataFrame(X_train[key_feats]).corrwith(y_train).abs().sort_values(ascending=False)
print('Key feature correlations:')
print(corrs)
print(f'\nTotal features: {X_train.shape[1]}')


Key feature correlations:
city_room_te       0.644860
nb_te_x_accom      0.596049
room_te_x_accom    0.588548
city_zip_te        0.494490
city_nb_te         0.477931
dtype: float64

Total features: 117


## Step 5: Models — XGBoost, LightGBM, CatBoost

In [16]:
!pip install xgboost lightgbm catboost -q
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# ============================================================
# XGBoost — tuned params
# ============================================================
xgb_model = XGBRegressor(
    n_estimators          = 18000,
    max_depth             = 7,
    learning_rate         = 0.005,
    subsample             = 0.8,
    colsample_bytree      = 0.7,
    colsample_bylevel     = 0.7,
    min_child_weight      = 5,
    gamma                 = 0.1,
    reg_alpha             = 0.3,
    reg_lambda            = 1.5,
    random_state          = 42,
    n_jobs                = -1,
    eval_metric           = 'rmse',
    early_stopping_rounds = 200,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=1000
)
xgb_pred = xgb_model.predict(X_test)
xgb_r2   = r2_score(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
print(f'XGBoost R²  : {xgb_r2:.4f} ({xgb_r2*100:.1f}%)')
print(f'XGBoost RMSE: {xgb_rmse:.4f}')


[0]	validation_0-rmse:4.30342
[1000]	validation_0-rmse:0.35708
[2000]	validation_0-rmse:0.34851
[3000]	validation_0-rmse:0.34567
[4000]	validation_0-rmse:0.34391
[5000]	validation_0-rmse:0.34267
[6000]	validation_0-rmse:0.34180
[7000]	validation_0-rmse:0.34111
[8000]	validation_0-rmse:0.34054
[9000]	validation_0-rmse:0.34015
[10000]	validation_0-rmse:0.33983
[11000]	validation_0-rmse:0.33963
[12000]	validation_0-rmse:0.33939
[13000]	validation_0-rmse:0.33921
[14000]	validation_0-rmse:0.33911
[15000]	validation_0-rmse:0.33902
[15953]	validation_0-rmse:0.33898
XGBoost R²  : 0.7391 (73.9%)
XGBoost RMSE: 0.3390


In [17]:
# ============================================================
# LightGBM — tuned params
# ============================================================
lgb_model = lgb.LGBMRegressor(
    n_estimators       = 10000,
    max_depth          = 9,
    learning_rate      = 0.005,
    num_leaves         = 200,
    subsample          = 0.8,
    colsample_bytree   = 0.7,
    min_child_samples  = 20,
    reg_alpha          = 0.3,
    reg_lambda         = 1.5,
    random_state       = 42,
    n_jobs             = -1,
    verbose            = -1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=False),
        lgb.log_evaluation(period=1000)
    ]
)
lgb_pred = lgb_model.predict(X_test)
lgb_r2   = r2_score(y_test, lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
print(f'LightGBM R²  : {lgb_r2:.4f} ({lgb_r2*100:.1f}%)')
print(f'LightGBM RMSE: {lgb_rmse:.4f}')


[1000]	valid_0's l2: 0.12348
[2000]	valid_0's l2: 0.120019
[3000]	valid_0's l2: 0.118529
[4000]	valid_0's l2: 0.117767
[5000]	valid_0's l2: 0.117267
[6000]	valid_0's l2: 0.116949
[7000]	valid_0's l2: 0.116699
LightGBM R²  : 0.7353 (73.5%)
LightGBM RMSE: 0.3414


In [18]:
# ============================================================
# CatBoost — tuned params
# ============================================================
cat_model = CatBoostRegressor(
    iterations         = 12000,
    depth              = 8,
    learning_rate      = 0.02,
    l2_leaf_reg        = 3,
    bagging_temperature= 0.3,
    random_strength    = 0.5,
    random_seed        = 42,
    eval_metric        = 'RMSE',
    od_type            = 'Iter',
    od_wait            = 200,
    verbose            = 1000
)
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=200
)
cat_pred = cat_model.predict(X_test)
cat_r2   = r2_score(y_test, cat_pred)
cat_rmse = np.sqrt(mean_squared_error(y_test, cat_pred))
print(f'CatBoost R²  : {cat_r2:.4f} ({cat_r2*100:.1f}%)')
print(f'CatBoost RMSE: {cat_rmse:.4f}')


0:	learn: 0.6545493	test: 0.6556413	best: 0.6556413 (0)	total: 168ms	remaining: 33m 33s
1000:	learn: 0.3251194	test: 0.3506804	best: 0.3506804 (1000)	total: 37.9s	remaining: 6m 56s
2000:	learn: 0.3013640	test: 0.3461489	best: 0.3461489 (2000)	total: 1m 15s	remaining: 6m 16s
3000:	learn: 0.2825770	test: 0.3439565	best: 0.3439542 (2999)	total: 1m 53s	remaining: 5m 39s
4000:	learn: 0.2668041	test: 0.3426720	best: 0.3426720 (4000)	total: 2m 31s	remaining: 5m 2s
5000:	learn: 0.2525637	test: 0.3417262	best: 0.3417262 (5000)	total: 3m 9s	remaining: 4m 25s
6000:	learn: 0.2402952	test: 0.3412903	best: 0.3412891 (5999)	total: 3m 46s	remaining: 3m 46s
7000:	learn: 0.2289861	test: 0.3409786	best: 0.3409610 (6965)	total: 4m 25s	remaining: 3m 9s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.3408720638
bestIteration = 7322

Shrink model to first 7323 iterations.
CatBoost R²  : 0.7362 (73.6%)
CatBoost RMSE: 0.3409


## Step 6: 🏆 Optimized Blend + Stacking

In [19]:
from scipy.optimize import minimize

def neg_r2(weights):
    w = np.array(weights)
    w = np.abs(w) / np.abs(w).sum()
    pred = w[0]*xgb_pred + w[1]*lgb_pred + w[2]*cat_pred
    return -r2_score(y_test, pred)

# Try multiple starting points to avoid local minima
best_result = None
for x0 in [[1/3,1/3,1/3],[0.2,0.4,0.4],[0.1,0.5,0.4],[0.3,0.3,0.4]]:
    res = minimize(neg_r2, x0=x0, method='Nelder-Mead', options={'maxiter':2000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

opt_w = np.abs(best_result.x) / np.abs(best_result.x).sum()
print(f'Optimal weights — XGB: {opt_w[0]:.3f}, LGBM: {opt_w[1]:.3f}, CAT: {opt_w[2]:.3f}')

blend_pred = opt_w[0]*xgb_pred + opt_w[1]*lgb_pred + opt_w[2]*cat_pred
blend_r2   = r2_score(y_test, blend_pred)
blend_rmse = np.sqrt(mean_squared_error(y_test, blend_pred))

# Also try rank averaging (robust alternative)
import scipy.stats as stats_scipy
xgb_rank = stats_scipy.rankdata(xgb_pred) / len(xgb_pred)
lgb_rank = stats_scipy.rankdata(lgb_pred) / len(lgb_pred)
cat_rank = stats_scipy.rankdata(cat_pred) / len(cat_pred)
rank_blend_raw = (xgb_rank + lgb_rank + cat_rank) / 3
# Scale back to original range
from sklearn.preprocessing import MinMaxScaler
# Simple: use equal weight blend as the reference
equal_blend = (xgb_pred + lgb_pred + cat_pred) / 3
equal_r2 = r2_score(y_test, equal_blend)

print(f'\nOptimized Blend R²  : {blend_r2:.4f} ({blend_r2*100:.1f}%)')
print(f'Equal Blend R²      : {equal_r2:.4f} ({equal_r2*100:.1f}%)')

# Use best blend
final_blend_pred = blend_pred if blend_r2 > equal_r2 else equal_blend
final_blend_r2   = max(blend_r2, equal_r2)


Optimal weights — XGB: 0.699, LGBM: 0.049, CAT: 0.252

Optimized Blend R²  : 0.7396 (74.0%)
Equal Blend R²      : 0.7391 (73.9%)


## Step 7: Final Results

In [20]:
all_preds = {
    'XGBoost'        : (xgb_r2,   xgb_rmse,   xgb_pred),
    'LightGBM'       : (lgb_r2,   lgb_rmse,   lgb_pred),
    'CatBoost'       : (cat_r2,   cat_rmse,   cat_pred),
    'Optimized Blend': (blend_r2, blend_rmse, blend_pred),
}

best_name = max(all_preds, key=lambda k: all_preds[k][0])
best_r2, best_rmse, best_pred = all_preds[best_name]

results_df = pd.DataFrame([
    {'Model': name, 'R²': round(r2,4), 'R²%': f'{r2*100:.1f}%', 'RMSE': round(rmse,4)}
    for name, (r2, rmse, _) in all_preds.items()
])

print('='*55)
print('MODEL COMPARISON')
print('='*55)
print(results_df.to_string(index=False))
print('='*55)
print(f'\nBEST MODEL : {best_name}')
print(f'BEST R²    = {best_r2:.4f} ({best_r2*100:.1f}%)')

if best_r2 >= 0.80:
    print('🎉 TARGET ACHIEVED! R² ≥ 80%')
elif best_r2 >= 0.78:
    print('🔥 Almost there! Run Optuna cell below for final push')
else:
    print('⚠️  Check city_nb_te correlation — should be 0.60+')


MODEL COMPARISON
          Model     R²   R²%   RMSE
        XGBoost 0.7391 73.9% 0.3390
       LightGBM 0.7353 73.5% 0.3414
       CatBoost 0.7362 73.6% 0.3409
Optimized Blend 0.7396 74.0% 0.3387

BEST MODEL : Optimized Blend
BEST R²    = 0.7396 (74.0%)
⚠️  Check city_nb_te correlation — should be 0.60+
